# 🚦 Smart Traffic Annotation System

**AI-powered traffic video annotation with real-time vehicle detection, tracking & analytics**

| Feature | Technology |
|---|---|
| **Detection** | YOLOv8 (Ultralytics) |
| **Tracking** | DeepSORT |
| **Backend** | FastAPI + SQLite |
| **Frontend** | HTML5 Canvas + JS |
| **Analytics** | Counting, Speed, Lane Analysis, Safety (TTC) |
| **Export** | COCO, YOLO, VOC, CSV, JSON |

> **⚠️ Run cells in order. Enable GPU:** `Runtime > Change runtime type > T4 GPU`


## 📋 Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 📦 Step 2: Install Dependencies

> **⚠️ IMPORTANT:** Wait for this cell to fully complete before running the next one!

In [ ]:
# Install all required packages
!pip install fastapi==0.104.1
!pip install 'uvicorn[standard]==0.24.0'
!pip install python-multipart==0.0.6
!pip install opencv-python==4.8.1.78
!pip install numpy==1.24.3
!pip install pillow==10.1.0
!pip install sqlalchemy==2.0.23
!pip install pydantic==2.5.0
!pip install python-dotenv==1.0.0
!pip install ultralytics
!pip install deep-sort-realtime
!pip install pyngrok

print('\n' + '='*50)
print('Verifying installations...')
print('='*50)

import ultralytics
print(f'ultralytics: {ultralytics.__version__}')

import fastapi
print(f'fastapi: {fastapi.__version__}')

import cv2
print(f'opencv: {cv2.__version__}')

from deep_sort_realtime.deepsort_tracker import DeepSort
print(f'deep-sort-realtime: OK')

from pyngrok import ngrok
print(f'pyngrok: OK')

print('\n\u2705 All packages installed and verified!')


## 🏗️ Step 3: Clone Repository

In [ ]:
import os

# Remove old clone if exists
!rm -rf /content/traffic-app

# Clone fresh
!git clone https://github.com/Dakshbumb/traffic-annotation-system.git /content/traffic-app

# Create required directories
os.makedirs('/content/traffic-app/backend/uploads', exist_ok=True)
os.makedirs('/content/traffic-app/backend/exports', exist_ok=True)

# Verify
os.chdir('/content/traffic-app/backend')
print(f'\nWorking dir: {os.getcwd()}')
print(f'Files: {[f for f in os.listdir(".") if f.endswith(".py")]}')
print('\u2705 Repository cloned!')


## 🤖 Step 4: Download YOLOv8 Model

In [ ]:
import os
os.chdir('/content/traffic-app/backend')

from ultralytics import YOLO
print('Downloading YOLOv8m model (this may take a minute)...')
model = YOLO('yolov8m.pt')
print(f'\u2705 Model ready! Size: {os.path.getsize("yolov8m.pt") / 1024 / 1024:.1f} MB')


## 🌐 Step 5: Setup ngrok Tunnel

To access the web UI from your browser, you need a free **ngrok** tunnel:

1. Sign up at [ngrok.com](https://ngrok.com) (free)
2. Go to [Your Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Copy & paste it below


In [ ]:
NGROK_TOKEN = input('Enter your ngrok authtoken: ')

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('\u2705 ngrok configured!')


## 🚀 Step 6: Start the Server

> After running this cell, click the **Public URL** link to open the web interface!

In [ ]:
import subprocess, time, os
import torch
from pyngrok import ngrok

os.chdir('/content/traffic-app/backend')

# Start FastAPI server
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
print('Starting server...')
time.sleep(8)

# Check server is running
if server.poll() is not None:
    print('\u274c Server failed to start! Error log:')
    print(server.stderr.read().decode())
else:
    # Create ngrok tunnel
    public_url = ngrok.connect(8000)
    
    print('=' * 60)
    print('\U0001f6a6 SMART TRAFFIC ANNOTATION SYSTEM')
    print('=' * 60)
    print(f'\n\U0001f310 Public URL: {public_url}')
    print(f'\U0001f4da API Docs:   {public_url}/docs')
    print(f'\u26a1 GPU: {"CUDA \u2705" if torch.cuda.is_available() else "CPU only"}')
    print('\n\U0001f4dd How to use:')
    print('   1. Click the Public URL above')
    print('   2. Upload a traffic video (MP4/AVI/MOV)')
    print('   3. Click Auto-Label for YOLOv8 + DeepSORT')
    print('   4. Use Edit / Analytics / Lanes / Safety modes')
    print('   5. Export in COCO / YOLO / VOC / CSV format')
    print('\n\u26a0\ufe0f  Keep this cell running!')
    print('=' * 60)


In [ ]:
# Keep server running - press STOP button to shut down
try:
    while True:
        time.sleep(60)
        if server.poll() is not None:
            print('Server stopped! Logs:')
            print(server.stderr.read().decode()[-3000:])
            break
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('Server shut down.')


## 📹 (Optional) Upload a Video from Your Computer

In [ ]:
from google.colab import files
import os

print('Select a traffic video file...')
uploaded = files.upload()
for fn in uploaded:
    dest = f'/content/traffic-app/backend/uploads/{fn}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fn])
    print(f'\u2705 Saved: {fn} ({len(uploaded[fn])/1024/1024:.1f} MB)')


## 📋 (Optional) Check Server Status

In [ ]:
if server.poll() is None:
    print('\u2705 Server is running')
else:
    print('\u274c Server has stopped')
    print('\nError log:')
    print(server.stderr.read().decode()[-2000:])
